# TF-IDF 트랙 전처리

**담당**: 영민  
**데이터**: AI Hub 낚시성 기사 탐지 데이터 1세부 (`work_pool`)  
**목표**: TF-IDF 벡터화를 위한 전처리 + 유형별 키워드 분석

---

## 전체 단계
1. **데이터 로딩** ✅
2. **텍스트 정제** ✅
3. **형태소 분석 (Komoran)** ✅
4. **불용어 제거** ✅ (\<NUM\>, 숫자 토큰 추가 처리)
5. **TF-IDF 벡터화** ✅
6. **키워드 분석** ✅ (Class-level TF-IDF)

In [17]:
import pandas as pd
import numpy as np
import re
import time

df = pd.read_parquet('data/processed/work_pool_tfidf_tokens.parquet')
print(f'불러오기 완료: {len(df):,}건')
print(f'컬럼: {list(df.columns)}')

불러오기 완료: 291,466건
컬럼: ['newsID', 'binary_label', 'type_label', 'source_class', 'title_clean', 'title_tfidf', 'title_morphs', 'title_tokens']


---

## Step 1-4. 불용어 제거 (최종 수정)

### 이전 결과에서 발견한 문제

`num`이 모든 클래스에서 1위로 나온 원인을 분석했다:

| 단계 | 실제 값 |
|---|---|
| `title_morphs`에 저장된 토큰 | `<NUM>` (꺽쇠 포함) |
| STOPWORDS에 넣은 값 | `num` (꺽쇠 없음) |
| 결과 | 매칭 실패 → `<NUM>` 필터 통과 → TF-IDF가 `num`으로 추출 |

**수정**: STOPWORDS 문자열 매칭 대신, `remove_stopwords` 함수에 규칙 2개 추가
- `tok.isdigit()` → `19` 같은 숫자만으로 된 토큰 제거
- `re.match(r'^<.+>$', tok)` → `<NUM>` 같은 꺽쇠 토큰 제거

In [18]:
STOPWORDS = {
    # 의존명사
    '것', '수', '들', '등', '때', '곳', '점', '뿐', '만큼', '정도', '경우',
    '상황', '입장', '가운데', '속', '중', '전', '후', '내', '위', '아래',
    '사이', '동안', '이상', '이하', '이후', '이전', '이번', '지난',
    # 매우 일반적인 명사
    '관련', '문제', '사실', '이유', '결과', '방법', '내용', '부분',
    '기자', '뉴스', '기사', '보도', '발표', '공개', '확인',
    '예정', '계획', '방침', '결정', '진행', '실시', '추진',
    '올해', '최근', '지금', '현재', '앞서', '이어', '또한', '함께',
    # 매우 일반적인 동사/형용사
    '있다', '없다', '하다', '되다', '이다', '아니다', '같다', '받다',
    '갖다', '나다', '보다', '오다', '가다', '알다', '말다',
    '크다', '작다', '많다', '적다', '좋다', '나쁘다',
    '통해', '위해', '대해', '따라', '따른', '의한', '위한', '통한',
    # 숫자 단위
    '년', '월', '일', '시', '분', '초', '개', '명', '원', '달러',
    '억', '조', '천', '백', '만',
    # 개선 추가
    '위하',  # 위하다(~을 위해) 어간, 변별력 없음
}

def remove_stopwords(token_str):
    """
    형태소 분석 결과에서 불필요한 토큰을 제거한다.
    
    제거 조건 (4가지 모두 통과해야 유지):
    1. 길이 2 이상
    2. 불용어 목록에 없음
    3. 숫자만으로 구성된 토큰 아님 ('19', '100' 등)
    4. 꺽쇠 토큰 아님 ('<NUM>' 등)
    """
    if not isinstance(token_str, str) or not token_str.strip():
        return ''
    tokens = token_str.split()
    filtered = [
        tok for tok in tokens
        if len(tok) >= 2                      # 조건 1: 1글자 제거
        and tok not in STOPWORDS              # 조건 2: 불용어 제거
        and not tok.isdigit()                 # 조건 3: 순수 숫자 토큰 제거
        and not re.match(r'^<.+>$', tok)      # 조건 4: <NUM> 등 꺽쇠 토큰 제거
    ]
    return ' '.join(filtered)

# 함수 동작 확인
test = '정부 경제 <NUM> 발표 충격 19 위하'
print(f'테스트 입력: {test}')
print(f'필터 결과:  {remove_stopwords(test)}')
print(f'  → <NUM>, 19, 위하 가 제거되어야 함')

테스트 입력: 정부 경제 <NUM> 발표 충격 19 위하
필터 결과:  정부 경제 충격
  → <NUM>, 19, 위하 가 제거되어야 함


In [19]:
print('전체 데이터 불용어 제거 중...')
t = time.time()
df['title_tokens'] = df['title_morphs'].apply(remove_stopwords)
print(f'완료! ({time.time()-t:.1f}초)')

before = df['title_morphs'].str.split().str.len().mean()
after  = df['title_tokens'].str.split().str.len().mean()
empty  = (df['title_tokens'] == '').sum()
print(f'평균 토큰 수: {before:.1f} → {after:.1f}개')
print(f'빈 토큰 케이스: {empty:,}건')

# 샘플 확인: <NUM>이 제대로 제거됐는지
sample = df[df['title_morphs'].str.contains('<NUM>', na=False)].head(3)
print('\n<NUM> 포함 케이스 필터 결과 확인:')
for _, row in sample.iterrows():
    print(f'  before: {row["title_morphs"]}')
    print(f'  after:  {row["title_tokens"]}')
    print()

전체 데이터 불용어 제거 중...
완료! (1.7초)
평균 토큰 수: 8.1 → 6.4개
빈 토큰 케이스: 125건

<NUM> 포함 케이스 필터 결과 확인:
  before: 오미 크 <NUM> 입원 당국 규제 계획 없어
  after:  오미 입원 당국 규제 없어

  before: 기후 변화 리스크 대하 <NUM> 오해
  after:  기후 변화 리스크 대하 오해

  before: 홍콩 경찰 정부 시위 연루 야당 국회의원 <NUM> 체포
  after:  홍콩 경찰 정부 시위 연루 야당 국회의원 체포



In [20]:
save_cols = ['newsID', 'binary_label', 'type_label', 'source_class',
             'title_clean', 'title_tfidf', 'title_morphs', 'title_tokens']
df[save_cols].to_parquet('data/processed/work_pool_tfidf_tokens.parquet', index=False)
print('저장 완료 ✅')

저장 완료 ✅


---

## Step 1-5. TF-IDF 벡터화

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = df['title_tokens'].fillna('').tolist()

print('TF-IDF 벡터화 시작...')
t = time.time()
vectorizer = TfidfVectorizer(max_features=10_000, min_df=5, max_df=0.95)
tfidf_matrix = vectorizer.fit_transform(corpus)
print(f'완료! ({time.time()-t:.1f}초)')
print(f'행렬 크기: {tfidf_matrix.shape[0]:,} × {tfidf_matrix.shape[1]:,}')
print(f'어휘 사전: {len(vectorizer.vocabulary_):,}개')
print(f'행렬 밀도: {tfidf_matrix.nnz/(tfidf_matrix.shape[0]*tfidf_matrix.shape[1])*100:.3f}%')

TF-IDF 벡터화 시작...
완료! (1.8초)
행렬 크기: 291,466 × 10,000
어휘 사전: 10,000개
행렬 밀도: 0.057%


---

## Step 1-6. 키워드 분석 (Class-level TF-IDF)

In [22]:
def class_tfidf_keywords(df, label_col, label_map, token_col='title_tokens', n=20):
    class_docs  = {}
    class_sizes = {}
    for label_id, label_name in label_map.items():
        mask = df[label_col] == label_id
        class_docs[label_name]  = ' '.join(df.loc[mask, token_col].fillna(''))
        class_sizes[label_name] = int(mask.sum())

    doc_list    = list(class_docs.values())
    class_names = list(class_docs.keys())

    cv   = TfidfVectorizer(max_features=20_000, min_df=1)
    mat  = cv.fit_transform(doc_list)
    feat = cv.get_feature_names_out()

    results = {}
    for i, cname in enumerate(class_names):
        scores  = mat[i].toarray().flatten()
        top_idx = scores.argsort()[::-1][:n]
        results[cname] = [(feat[j], scores[j]) for j in top_idx]
    return results, class_sizes

print('class_tfidf_keywords 함수 정의 완료')

class_tfidf_keywords 함수 정의 완료


In [23]:
binary_map = {0: '정상', 1: '낚시성'}
binary_kw, binary_sizes = class_tfidf_keywords(df, 'binary_label', binary_map, n=20)

clickbait_kw = binary_kw['낚시성']
normal_kw    = binary_kw['정상']

print('='*58)
print(f'{"낚시성 기사 특징 키워드 TOP20":^27} | {"정상 기사 특징 키워드 TOP20":^27}')
print('='*58)
for (cw, cs), (nw, ns) in zip(clickbait_kw, normal_kw):
    print(f'{cw:>15} ({cs:.4f})  |  {nw:<15} ({ns:.4f})')

    낚시성 기사 특징 키워드 TOP20     |     정상 기사 특징 키워드 TOP20     
            코로나 (0.2596)  |  코로나             (0.2461)
             정부 (0.1568)  |  지원              (0.1562)
             기업 (0.1400)  |  출시              (0.1401)
             시장 (0.1327)  |  기업              (0.1268)
             한국 (0.1324)  |  한국              (0.1259)
            대통령 (0.1315)  |  정부              (0.1258)
             백신 (0.1313)  |  시장              (0.1198)
             논란 (0.1236)  |  대통령             (0.1174)
             미국 (0.1216)  |  사업              (0.1173)
             중국 (0.1189)  |  확진              (0.1105)
             지역 (0.1091)  |  백신              (0.1054)
             세계 (0.1077)  |  트럼프             (0.0957)
             지원 (0.1048)  |  최대              (0.0942)
             확진 (0.1016)  |  세계              (0.0918)
             사람 (0.0989)  |  국민              (0.0912)
             사업 (0.0985)  |  논란              (0.0908)
             국민 (0.0953)  |  만원              (0.0907)
             종합 (0.0950)

In [24]:
type_map = {
    0: '의문유발-부호(11)', 1: '의문유발-은닉(12)', 2: '선정표현(13)',
    3: '속어/줄임말(14)',   4: '사실과대(15)',      5: '주어왜곡(16)'
}
direct_df = df[df['type_label'] != -1].copy()
type_kw, type_sizes = class_tfidf_keywords(direct_df, 'type_label', type_map, n=15)

print('=== 낚시성 유형별 특징 키워드 TOP 15 (Class-level TF-IDF) ===')
for type_name, keywords in type_kw.items():
    kw_str = ', '.join([f'{w}({s:.3f})' for w, s in keywords])
    print(f'\n[{type_name}] ({type_sizes[type_name]:,}건)')
    print(f'  {kw_str}')

=== 낚시성 유형별 특징 키워드 TOP 15 (Class-level TF-IDF) ===

[의문유발-부호(11)] (14,863건)
  코로나(0.255), 미국(0.209), 어떻(0.173), 중국(0.172), 정부(0.159), 대통령(0.158), 시장(0.151), 한국(0.151), 기업(0.140), 일본(0.115), 지역(0.110), 논란(0.110), 국내(0.106), 백신(0.103), 사업(0.103)

[의문유발-은닉(12)] (11,685건)
  사람(0.424), 코로나(0.221), 미국(0.209), 정부(0.166), 기업(0.153), 중국(0.150), 밝히(0.144), 대통령(0.134), 지역(0.133), 논란(0.132), 배우(0.107), 한국(0.104), 나라(0.101), 사건(0.100), 시장(0.098)

[선정표현(13)] (3,554건)
  충격(0.247), 코로나(0.235), 파격(0.233), 신음(0.227), 달아오르(0.221), 경악(0.193), 발가벗기(0.182), 노골적(0.171), 정부(0.147), 튀기(0.133), 중국(0.132), 미국(0.125), 속살(0.123), 논란(0.122), 한국(0.120)

[속어/줄임말(14)] (3,696건)
  코로나(0.198), 미국(0.175), 빠꾸(0.169), 기업(0.150), 시장(0.142), 대통령(0.141), 코시(0.135), 정부(0.133), 중국(0.130), 사업(0.117), 캐리(0.116), 멘붕(0.113), 한국(0.112), 버프(0.108), 극대(0.108)

[사실과대(15)] (4,228건)
  세계(0.251), 최고(0.222), 코로나(0.217), 역대(0.200), 국민(0.175), 정부(0.174), 중국(0.161), 미국(0.157), 대통령(0.135), 한국(0.129), 최악(0.127), 시장(0.118), 기업(0.107), 국내(0.106), 

In [25]:
import os
print('='*60)
print('TF-IDF 트랙 전처리 전 단계 완료!')
print('='*60)
fsize = os.path.getsize('data/processed/work_pool_tfidf_tokens.parquet')/1024/1024
print(f'\n저장 파일: data/processed/work_pool_tfidf_tokens.parquet ({fsize:.1f} MB)')
print(f'tfidf_matrix: {tfidf_matrix.shape} 희소행렬')
print(f'vectorizer  : 어휘 {len(vectorizer.vocabulary_):,}개')
print('\n[다음 단계]')
print('  → tfidf_matrix로 분류 모델 학습 (Logistic Regression, SVM 등)')
print('  → 5-Fold Stratified Cross-Validation')

TF-IDF 트랙 전처리 전 단계 완료!

저장 파일: data/processed/work_pool_tfidf_tokens.parquet (58.4 MB)
tfidf_matrix: (291466, 10000) 희소행렬
vectorizer  : 어휘 10,000개

[다음 단계]
  → tfidf_matrix로 분류 모델 학습 (Logistic Regression, SVM 등)
  → 5-Fold Stratified Cross-Validation
